In [1]:
import json
import os
import numpy as np
import pandas as pd
import torch
from momentfm import MOMENTPipeline


import time

start_time = time.time()

def root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

# ============================================================
# CONFIG
# ============================================================
DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR  = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR   = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"

countries = ["Germany", "Ireland", "Portugal"]
days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation"
]

countries = ["Denmark"]

days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
    "price_eur_kwh"
]

PREDICTION_LENGTH = 96
MODEL_NAME = "AutonLab/MOMENT-1-base"   # or "AutonLab/MOMENT-1-large"

# ============================================================
# HELPERS
# ============================================================
def prepare_moment_input(series: pd.Series, seq_len: int):
    """
    Convert a univariate pandas Series into MOMENT input:
    x_enc:      [1, 1, seq_len]
    input_mask: [1, seq_len]

    If series is shorter than seq_len, left-pad with zeros and mask padded values with 0.
    If series is longer than seq_len, keep only the last seq_len values.
    """
    values = series.astype(float).to_numpy()

    # Replace inf/nan safely
    values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)

    if len(values) >= seq_len:
        x = values[-seq_len:]
        mask = np.ones(seq_len, dtype=np.float32)
    else:
        pad_len = seq_len - len(values)
        x = np.concatenate([np.zeros(pad_len, dtype=np.float32), values.astype(np.float32)])
        mask = np.concatenate([np.zeros(pad_len, dtype=np.float32), np.ones(len(values), dtype=np.float32)])

    # Shape for MOMENT: [batch, n_channels, seq_len]
    x_enc = torch.tensor(x, dtype=torch.float32).unsqueeze(0).unsqueeze(0)   # [1, 1, seq_len]
    input_mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)        # [1, seq_len]

    return x_enc, input_mask


# ============================================================
# LOAD SPLITS
# ============================================================
with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

# ============================================================
# LOAD MODEL
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = MOMENTPipeline.from_pretrained(
    MODEL_NAME,
    model_kwargs={
        "task_name": "forecasting",
        "forecast_horizon": PREDICTION_LENGTH,
    },
)
model.init()
model = model.to(device)
model.eval()

# IMPORTANT:
# MOMENT forecasting head expects a fixed input length consistent with model config.
seq_len = model.config.seq_len
print("MOMENT seq_len:", seq_len)

# ============================================================
# FORECAST LOOP
# ============================================================
rmse_results = []

for country in countries:
    print(f"\nProcessing country: {country}")

    data_path = rf"{DATA_DIR}\dataset_{country.capitalize()}.csv"
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    households = [col for col in df.columns if col not in features]

    for day in days:
        print(f"   Day: {day}")

        cutoff = pd.to_datetime(dataset_days[country][day])

        # Ground-truth horizon timestamps
        future_index = df.index[df.index >= cutoff][:PREDICTION_LENGTH]

        if len(future_index) < PREDICTION_LENGTH:
            print(f"      Skipping {country}-{day}: not enough future points ({len(future_index)})")
            continue

        predictions_df_all_households = pd.DataFrame(index=future_index)
        rmse_households = []

        for household in households:
            # Training history up to cutoff
            s_train = df.loc[df.index < cutoff, household].dropna()

            if len(s_train) == 0:
                print(f"      Skipping household {household}: no training data")
                continue

            # Build MOMENT input
            x_enc, input_mask = prepare_moment_input(s_train, seq_len=seq_len)
            x_enc = x_enc.to(device)
            input_mask = input_mask.to(device)

            # Predict
            with torch.no_grad():
                outputs = model(x_enc=x_enc, input_mask=input_mask)
                y_pred = outputs.forecast.squeeze(0).squeeze(0).detach().cpu().numpy()

            # Keep exactly prediction horizon length
            y_pred = y_pred[:PREDICTION_LENGTH]

            # Store predictions
            predictions_df_all_households[household] = y_pred

            # RMSE
            y_true = df.loc[future_index, household].to_numpy(dtype=float)
            rmse = root_mean_squared_error(y_true, y_pred)
            rmse_households.append(rmse)

        if len(rmse_households) == 0:
            print(f"      No valid household forecasts for {country}-{day}")
            continue

        avg_rmse_households = float(np.mean(rmse_households))

        rmse_results.append({
            "country": country,
            "day": day,
            "rmse": avg_rmse_households
        })

        # Save predictions
        output = rf"{OUT_DIR}\MOMENT_Univar_pred_{day}_{country.capitalize()}.csv"
        os.makedirs(os.path.dirname(output), exist_ok=True)
        predictions_df_all_households.to_csv(output, index=True)
        print("      Saved:", output)

# ============================================================
# SUMMARY
# ============================================================
rmse_df = pd.DataFrame(rmse_results)

print("\nPer-day RMSE:")
print(rmse_df)

print("\nCross-validated RMSE per country (mean over days):")
print(rmse_df.groupby("country")["rmse"].mean())

end_time = time.time()
total_seconds = end_time - start_time
print(f"Total runtime: {total_seconds:.2f} seconds")

c:\Users\CR58XM\AppData\Local\anaconda3\envs\moment\Lib\site-packages\transformers\utils\generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


Using device: cuda


c:\Users\CR58XM\AppData\Local\anaconda3\envs\moment\Lib\site-packages\momentfm\models\moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


MOMENT seq_len: 512

Processing country: Denmark
   Day: day1
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\MOMENT_Univar_pred_day1_Denmark.csv
   Day: day2
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\MOMENT_Univar_pred_day2_Denmark.csv
   Day: day3
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\MOMENT_Univar_pred_day3_Denmark.csv
   Day: day4
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\MOMENT_Univar_pred_day4_Denmark.csv
   Day: day5
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\MOMENT_Univar_pred_day5_Denmark.csv

Per-day RMSE:
   country   day      rmse
0  Denmark  day1  1.988318
1  Denmark  day2  0.723279
2  Denmark  day3  0.602197
3  Denmark  day4  0.478676
4  Denmark  day5  1.273364

Cross-validated RMSE per country (mean over days

In [2]:
print(f"Time taken: {total_seconds:.4f} seconds")


file_path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\time_spend.json"
# 1. Load existing JSON
with open(file_path, "r") as f:
    data = json.load(f)

# 2. Add model inside "Local"
data["Foundational"]["Moment"] = total_seconds

# 3. Save back (without disturbing structure)
with open(file_path, "w") as f:
    json.dump(data, f, indent=4)

Time taken: 3.5248 seconds
